In [ ]:
import torch
import torch.nn as nn
from monai.networks.nets import resnet18
# Daha önce yazdığımız mimari sınıfının aynısı
from model_architecture import RSNAResNetCBAMClassifier 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Boş modeli tanımla
model = RSNAResNetCBAMClassifier(num_classes=13)

# 2. Kaggle'dan indirdiğin ağırlıkları yükle
WEIGHTS_PATH = "rsna2023_best_attention_model.pth"
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
print("RSNA Ön eğitim ağırlıkları başarıyla yüklendi!")

# 3. KATMANLARI FREEZE ETME (Dondurma)
# ResNet omurgasındaki tüm gradyan takibini kapatıyoruz
for param in model.backbone.parameters():
    param.requires_grad = False

# İsteğe bağlı: CBAM dikkat bloklarını açık bırakabilirsin ki TR_ABDOMEN'e göre odağını esnetebilsin
# Sadece son lineer katmanı eğiteceksek (Linear Probe):
for param in model.custom_classifier.parameters():
    param.requires_grad = True

# 4. TR_ABDOMEN Sınıf Sayısına Göre Son Katmanı Güncelleme
# Eğer TR_ABDOMEN veri setindeki sınıf sayısı RSNA'den farklıysa (Örn: 5 sınıf olsun):
TR_NUM_CLASSES = 5 
model.custom_classifier = nn.Linear(512, TR_NUM_CLASSES)

model = model.to(device)

In [ ]:

import matplotlib.pyplot as plt

from model_architecture import GradCAM3D

# --- GRAD-CAM UYGULAMA VE GÖRSELLEŞTİRME ---

# ResNet18'in en derin evrişimsel katmanını (Layer 4) hedef seçiyoruz
target_layer = model.backbone.layer4

gcam = GradCAM3D(model, target_layer)

# Tek bir test hastası seçelim (Validation setinden)
# input_tensor boyutu: [1, 1, 128, 128, 128] olmalı
images, labels = next(iter(val_loader))
input_tensor = images[0:1].to(device)

# Modelin en yüksek olasılık verdiği sınıf (veya analiz etmek istediğin hasar sınıfı)
predicted_class = 0 

# Isı haritasını üret (3D array döner: 128x128x128)
heatmap_3d = gcam.generate_heatmap(input_tensor, class_idx=predicted_class)

# --- 2D KESİT GÖRSELLEŞTİRME ---
# 3D hacmin tam ortasındaki (64. slice) kesiti ekrana basalım
slice_idx = 64

raw_slice = input_tensor[0, 0, :, :, slice_idx].cpu().numpy()
cam_slice = heatmap_3d[:, :, slice_idx]

plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(raw_slice, cmap="bone")
plt.title("Ham CT Kesiti (TR_ABDOMEN)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(cam_slice, cmap="jet")
plt.title("Grad-CAM Aktivasyon Haritası")
plt.axis("off")

plt.subplot(1, 3, 3)
# Ham görüntü üzerine ısı haritasını bindiriyoruz (%50 şeffaflıkla)
plt.imshow(raw_slice, cmap="bone")
plt.imshow(cam_slice, cmap="jet", alpha=0.5)
plt.title("Birleştirilmiş (Overlay) Görüntü")
plt.axis("off")

plt.show()